In [ ]:
# Install optional dependencies (run in Colab)
# Uncomment line below if pycocotools is missing.
# %pip install -q pycocotools

In [ ]:
# Colab setup (safe to run on local too)
from pathlib import Path
import importlib

DRIVE_ROOT = Path('/content/drive/MyDrive')

drive_mod = importlib.import_module('google.colab.drive')
drive_mod.mount('/content/drive', force_remount=False)
print('Running on Colab. Drive mounted at /content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Running on Colab. Drive mounted at /content/drive


## Colab Quick Setup

Chạy 2 cell đầu tiên bên dưới khi dùng Google Colab:
1. Mount Google Drive.
2. Cài dependencies.
3. Tự động map đường dẫn dữ liệu trong Drive.

# MFEFNet Pipeline (VisDrone + UAVDT)

Notebook này triển khai pipeline đầy đủ theo hướng bám sát paper:
1. Chuẩn bị dữ liệu và chuyển đổi annotation sang YOLO.
2. Thiết kế kiến trúc MFEFNet:
   - Backbone CSP-like.
   - SIEM theo công thức Cat + attention map + residual + MaxPool 5x5.
   - GAFN với AFFM trọng số động theo nội dung từng ảnh.
   - Detection head đa tỉ lệ kiểu IDetect-like.
3. Loss với MPDIoU + BCE theo trọng số 0.3 / 0.05 / 0.7.
4. Cấu hình train: SGD, 300 epochs, lr 0.01, bs 16, wd 5e-4, momentum 0.937.
5. Evaluate COCO đầy đủ: mAP50, mAP50:95, APS, APM, APL.

In [ ]:
import importlib
import importlib.util
import json
import math
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Optional package availability check for evaluation
HAS_COCO = importlib.util.find_spec('pycocotools') is not None

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
# Enable TF32 on Ampere+ GPUs (e.g., A100) for much faster FP32 matmul/conv.
torch.backends.cudnn.allow_tf32 = True
torch.backends.cuda.matmul.allow_tf32 = True

print('Torch:', torch.__version__, '| Device:', DEVICE)
print('pycocotools available:', HAS_COCO)

Torch: 2.10.0+cu128 | Device: cuda
pycocotools available: True


In [ ]:
@dataclass
class Paths:
    root: Path
    visdrone_train: Path
    visdrone_val: Path
    visdrone_test: Path
    uavdt_train: Path
    uavdt_test: Path
    workdir: Path


def _looks_like_dataset_root(root: Path) -> bool:
    return (root / 'VisDrone').exists() and (root / 'UAVDT').exists()


def resolve_dataset_root() -> Path:
    """Resolve dataset root automatically for local or Colab."""
    candidates = [
        Path('/content/drive/MyDrive/UAV'),
        Path('/content/drive/MyDrive/UAV'),
        Path('/content/UAV'),
        Path('/content/drive/MyDrive'),
    ]

    # Direct matches
    for c in candidates:
        if _looks_like_dataset_root(c):
            return c

    # One-level nested matches (common when folder was uploaded with extra wrapper)
    for c in candidates:
        if not c.exists():
            continue
        try:
            for child in c.iterdir():
                if child.is_dir() and _looks_like_dataset_root(child):
                    return child
        except Exception:
            continue

    # Fallback to local default path style.
    return Path('d:/UAV')


def build_paths(root: Optional[Path] = None) -> Paths:
    root = root or resolve_dataset_root()
    return Paths(
        root=root,
        visdrone_train=root / 'VisDrone' / 'VisDrone2019-DET-train' / 'VisDrone2019-DET-train',
        visdrone_val=root / 'VisDrone' / 'VisDrone2019-DET-val' / 'VisDrone2019-DET-val',
        visdrone_test=root / 'VisDrone' / 'VisDrone2019-DET-test-dev' / 'VisDrone2019-DET-test-dev',
        uavdt_train=root / 'UAVDT' / 'train',
        uavdt_test=root / 'UAVDT' / 'test',
        workdir=root / 'processed',
    )


P = build_paths()
P.workdir.mkdir(parents=True, exist_ok=True)
print('Using dataset root:', P.root)

VISDRONE_NAMES = [
    'pedestrian', 'people', 'bicycle', 'car', 'van',
    'truck', 'tricycle', 'awning-tricycle', 'bus', 'motor'
]
UAVDT_NAMES = ['car', 'bus', 'truck']
UAVDT_NAME_TO_ID = {n: i for i, n in enumerate(UAVDT_NAMES)}


def head(path: Path, n: int = 5) -> List[str]:
    out: List[str] = []
    with path.open('r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if i >= n:
                break
            out.append(line.strip())
    return out


def inspect_raw_structure() -> None:
    vis_ann = next((P.visdrone_train / 'annotations').glob('*.txt'), None)
    uav_ann = next((P.uavdt_test / 'ann').glob('*.json'), None)

    if vis_ann is None:
        print('Warning: No VisDrone annotation found at', P.visdrone_train / 'annotations')
    else:
        print('VisDrone annotation sample:', vis_ann.name)
        print('\n'.join(head(vis_ann, 8)))

    if uav_ann is None:
        print('Warning: No UAVDT annotation found at', P.uavdt_test / 'ann')
    else:
        print('\nUAVDT annotation sample:', uav_ann.name)
        obj = json.loads(uav_ann.read_text(encoding='utf-8'))
        print('Keys:', list(obj.keys()))
        print('Image size:', obj.get('size'))
        print('First object keys:', list(obj['objects'][0].keys()) if obj.get('objects') else None)
        if obj.get('objects'):
            print('First object class:', obj['objects'][0].get('classTitle'))
            print('First object points:', obj['objects'][0].get('points'))

Using dataset root: /content/drive/MyDrive/UAV


In [ ]:
def xyxy_to_yolo(x1: float, y1: float, x2: float, y2: float, w: int, h: int) -> Tuple[float, float, float, float]:
    bw = max(0.0, x2 - x1)
    bh = max(0.0, y2 - y1)
    cx = x1 + bw / 2.0
    cy = y1 + bh / 2.0
    return cx / w, cy / h, bw / w, bh / h


def convert_visdrone_split(src: Path, dst: Path) -> Tuple[int, int]:
    img_src = src / 'images'
    ann_src = src / 'annotations'
    img_dst = dst / 'images'
    lbl_dst = dst / 'labels'
    img_dst.mkdir(parents=True, exist_ok=True)
    lbl_dst.mkdir(parents=True, exist_ok=True)

    n_img, n_box = 0, 0
    for ann in ann_src.glob('*.txt'):
        img = img_src / ann.with_suffix('.jpg').name
        if not img.exists():
            continue

        # Symlink/copy strategy: keep lightweight by storing path lists; labels are materialized.
        out_lbl = lbl_dst / ann.name
        rows = [r.strip().split(',') for r in ann.read_text(encoding='utf-8').splitlines() if r.strip()]

        with Image.open(img) as im:
            w, h = im.size

        lines = []
        for r in rows:
            x, y, bw, bh, _, cls, _, _ = map(int, r[:8])
            if cls == 0:
                continue  # ignore region
            cls_id = cls - 1
            x1, y1 = x, y
            x2, y2 = x + bw, y + bh
            cx, cy, nw, nh = xyxy_to_yolo(x1, y1, x2, y2, w, h)
            if nw <= 0 or nh <= 0:
                continue
            lines.append(f"{cls_id} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")
            n_box += 1

        out_lbl.write_text('\n'.join(lines), encoding='utf-8')
        n_img += 1

    return n_img, n_box


def convert_uavdt_split(src: Path, dst: Path) -> Tuple[int, int]:
    ann_src = src / 'ann'
    lbl_dst = dst / 'labels'
    lbl_dst.mkdir(parents=True, exist_ok=True)

    n_img, n_box = 0, 0
    for ann in ann_src.glob('*.json'):
        data = json.loads(ann.read_text(encoding='utf-8'))
        w = int(data['size']['width'])
        h = int(data['size']['height'])

        lines = []
        for obj in data.get('objects', []):
            cls_name = obj.get('classTitle', '').lower()
            if cls_name not in UAVDT_NAME_TO_ID:
                continue  # keep only car, bus, truck
            p = obj['points']['exterior']
            (x1, y1), (x2, y2) = p[0], p[1]
            cx, cy, nw, nh = xyxy_to_yolo(float(x1), float(y1), float(x2), float(y2), w, h)
            if nw <= 0 or nh <= 0:
                continue
            lines.append(f"{UAVDT_NAME_TO_ID[cls_name]} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")
            n_box += 1

        out_lbl = lbl_dst / ann.with_suffix('.txt').name
        out_lbl.write_text('\n'.join(lines), encoding='utf-8')
        n_img += 1

    return n_img, n_box


def build_yolo_lists(img_dir: Path, out_txt: Path) -> int:
    imgs = sorted(list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png')))
    out_txt.write_text('\n'.join(str(p.as_posix()) for p in imgs), encoding='utf-8')
    return len(imgs)


def prepare_data() -> Dict[str, int]:
    root = P.workdir
    vis = root / 'visdrone'
    uav = root / 'uavdt'

    # VisDrone split conversion
    vt, vb = convert_visdrone_split(P.visdrone_train, vis / 'train')
    vv, vb2 = convert_visdrone_split(P.visdrone_val, vis / 'val')
    vte, vb3 = convert_visdrone_split(P.visdrone_test, vis / 'test')

    # UAVDT split conversion (dataset currently has train/test folders in this workspace)
    ut, ub = convert_uavdt_split(P.uavdt_train, uav / 'train')
    ute, ub2 = convert_uavdt_split(P.uavdt_test, uav / 'val')

    # Build list files from image folders (reuse original images)
    build_yolo_lists(P.visdrone_train / 'images', vis / 'train_images.txt')
    build_yolo_lists(P.visdrone_val / 'images', vis / 'val_images.txt')
    build_yolo_lists(P.visdrone_test / 'images', vis / 'test_images.txt')

    build_yolo_lists(P.uavdt_train / 'img', uav / 'train_images.txt')
    build_yolo_lists(P.uavdt_test / 'img', uav / 'val_images.txt')

    # YAML files for training scripts
    (vis / 'visdrone.yaml').write_text(
        'path: ' + str(vis.as_posix()) + '\n'
        'train: train_images.txt\n'
        'val: val_images.txt\n'
        'test: test_images.txt\n'
        'names:\n' + '\n'.join([f'  {i}: {n}' for i, n in enumerate(VISDRONE_NAMES)]) + '\n',
        encoding='utf-8'
    )

    (uav / 'uavdt.yaml').write_text(
        'path: ' + str(uav.as_posix()) + '\n'
        'train: train_images.txt\n'
        'val: val_images.txt\n'
        'names:\n' + '\n'.join([f'  {i}: {n}' for i, n in enumerate(UAVDT_NAMES)]) + '\n',
        encoding='utf-8'
    )

    return {
        'vis_train_img': vt,
        'vis_val_img': vv,
        'vis_test_img': vte,
        'vis_boxes': vb + vb2 + vb3,
        'uav_train_img': ut,
        'uav_val_img': ute,
        'uav_boxes': ub + ub2,
    }

In [ ]:
def letterbox(img: Image.Image, size: int = 640, color: Tuple[int, int, int] = (114, 114, 114)) -> Tuple[Image.Image, float, Tuple[int, int]]:
    w, h = img.size
    scale = min(size / w, size / h)
    nw, nh = int(round(w * scale)), int(round(h * scale))
    resized = img.resize((nw, nh), Image.BILINEAR)

    canvas = Image.new('RGB', (size, size), color)
    pad_w, pad_h = (size - nw) // 2, (size - nh) // 2
    canvas.paste(resized, (pad_w, pad_h))
    return canvas, scale, (pad_w, pad_h)


def _yolo_to_xyxy_abs(labels: np.ndarray, w: int, h: int) -> np.ndarray:
    if labels.size == 0:
        return np.zeros((0, 5), dtype=np.float32)
    out = labels.copy().astype(np.float32)
    cx = out[:, 1] * w
    cy = out[:, 2] * h
    bw = out[:, 3] * w
    bh = out[:, 4] * h
    out[:, 1] = cx - bw / 2.0
    out[:, 2] = cy - bh / 2.0
    out[:, 3] = cx + bw / 2.0
    out[:, 4] = cy + bh / 2.0
    return out


def _xyxy_abs_to_yolo(labels_xyxy: np.ndarray, size: int) -> np.ndarray:
    if labels_xyxy.size == 0:
        return np.zeros((0, 5), dtype=np.float32)
    out = labels_xyxy.copy().astype(np.float32)
    x1, y1, x2, y2 = out[:, 1], out[:, 2], out[:, 3], out[:, 4]
    bw = (x2 - x1).clip(min=0)
    bh = (y2 - y1).clip(min=0)
    cx = x1 + bw / 2.0
    cy = y1 + bh / 2.0

    out[:, 1] = (cx / size).clip(0.0, 1.0)
    out[:, 2] = (cy / size).clip(0.0, 1.0)
    out[:, 3] = (bw / size).clip(0.0, 1.0)
    out[:, 4] = (bh / size).clip(0.0, 1.0)
    return out


class YoloTxtDataset(Dataset):
    def __init__(
        self,
        image_list_file: Path,
        labels_dir: Path,
        img_size: int = 640,
        train: bool = True,
        mosaic_prob: float = 0.8,
    ):
        raw_paths = [Path(x.strip()) for x in image_list_file.read_text(encoding='utf-8').splitlines() if x.strip()]
        self.labels_dir = labels_dir
        self.img_size = img_size
        self.train = train
        self.mosaic_prob = mosaic_prob

        self._filename_to_path: Dict[str, Path] = {}
        self._build_filename_index()
        self.img_paths = [self._resolve_image_path(p) for p in raw_paths]

        missing = [str(p) for p in self.img_paths if not p.exists()]
        if missing:
            preview = '\n'.join(missing[:3])
            raise FileNotFoundError(
                'Some image paths in image list do not exist after auto-remap. '
                'If you copied only processed/, please also copy raw image folders VisDrone and UAVDT, '
                'or regenerate processed on Colab with prepare_data().\n'
                f'Missing examples:\n{preview}'
            )

    def _candidate_image_dirs(self) -> List[Path]:
        dirs: List[Path] = []
        p_obj = globals().get('P', None)
        if p_obj is None:
            return dirs

        dirs.extend([
            p_obj.visdrone_train / 'images',
            p_obj.visdrone_val / 'images',
            p_obj.visdrone_test / 'images',
            p_obj.uavdt_train / 'img',
            p_obj.uavdt_test / 'img',
        ])
        return [d for d in dirs if d.exists()]

    def _build_filename_index(self) -> None:
        for d in self._candidate_image_dirs():
            for pattern in ('*.jpg', '*.png', '*.jpeg', '*.JPG', '*.PNG', '*.JPEG'):
                for p in d.glob(pattern):
                    self._filename_to_path.setdefault(p.name, p)

    def _resolve_image_path(self, p: Path) -> Path:
        if p.exists():
            return p
        by_name = self._filename_to_path.get(p.name)
        if by_name is not None and by_name.exists():
            return by_name
        return p

    def __len__(self) -> int:
        return len(self.img_paths)

    def _read_image_and_labels(self, idx: int) -> Tuple[Image.Image, np.ndarray]:
        p = self.img_paths[idx]
        if not p.exists():
            raise FileNotFoundError(f'Image not found: {p}')
        img = Image.open(p).convert('RGB')

        label_path = self.labels_dir / (p.stem + '.txt')
        labels = []
        if label_path.exists():
            for ln in label_path.read_text(encoding='utf-8').splitlines():
                c, cx, cy, w, h = ln.split()
                labels.append([int(c), float(cx), float(cy), float(w), float(h)])
        arr = np.asarray(labels, dtype=np.float32) if labels else np.zeros((0, 5), dtype=np.float32)
        return img, arr

    def _letterbox_with_labels(self, img: Image.Image, labels: np.ndarray) -> Tuple[Image.Image, np.ndarray]:
        w0, h0 = img.size
        lb, scale, (pad_w, pad_h) = letterbox(img, self.img_size)

        if labels.size == 0:
            return lb, labels

        lxyxy = _yolo_to_xyxy_abs(labels, w0, h0)
        lxyxy[:, 1] = lxyxy[:, 1] * scale + pad_w
        lxyxy[:, 2] = lxyxy[:, 2] * scale + pad_h
        lxyxy[:, 3] = lxyxy[:, 3] * scale + pad_w
        lxyxy[:, 4] = lxyxy[:, 4] * scale + pad_h

        lxyxy[:, 1:] = np.clip(lxyxy[:, 1:], 0.0, float(self.img_size))
        w_box = lxyxy[:, 3] - lxyxy[:, 1]
        h_box = lxyxy[:, 4] - lxyxy[:, 2]
        keep = (w_box > 1.0) & (h_box > 1.0)
        lxyxy = lxyxy[keep]

        return lb, _xyxy_abs_to_yolo(lxyxy, self.img_size)

    def _load_mosaic(self, idx: int) -> Tuple[Image.Image, np.ndarray]:
        s = self.img_size
        mosaic_img = np.full((s * 2, s * 2, 3), 114, dtype=np.uint8)
        yc = random.randint(s // 2, int(1.5 * s))
        xc = random.randint(s // 2, int(1.5 * s))

        indices = [idx] + random.choices(range(len(self.img_paths)), k=3)
        labels_all = []

        for i, index in enumerate(indices):
            img, labels = self._read_image_and_labels(index)
            w, h = img.size
            img_np = np.asarray(img, dtype=np.uint8)
            labels_xyxy = _yolo_to_xyxy_abs(labels, w, h)

            if i == 0:  # top-left
                x1a, y1a, x2a, y2a = max(xc - w, 0), max(yc - h, 0), xc, yc
                x1b, y1b, x2b, y2b = w - (x2a - x1a), h - (y2a - y1a), w, h
            elif i == 1:  # top-right
                x1a, y1a, x2a, y2a = xc, max(yc - h, 0), min(xc + w, 2 * s), yc
                x1b, y1b, x2b, y2b = 0, h - (y2a - y1a), min(w, x2a - x1a), h
            elif i == 2:  # bottom-left
                x1a, y1a, x2a, y2a = max(xc - w, 0), yc, xc, min(yc + h, 2 * s)
                x1b, y1b, x2b, y2b = w - (x2a - x1a), 0, w, min(y2a - y1a, h)
            else:  # bottom-right
                x1a, y1a, x2a, y2a = xc, yc, min(xc + w, 2 * s), min(yc + h, 2 * s)
                x1b, y1b, x2b, y2b = 0, 0, min(w, x2a - x1a), min(h, y2a - y1a)

            mosaic_img[y1a:y2a, x1a:x2a] = img_np[y1b:y2b, x1b:x2b]

            if labels_xyxy.size > 0:
                pad_x = x1a - x1b
                pad_y = y1a - y1b
                labels_xyxy[:, 1] += pad_x
                labels_xyxy[:, 2] += pad_y
                labels_xyxy[:, 3] += pad_x
                labels_xyxy[:, 4] += pad_y
                labels_all.append(labels_xyxy)

        if labels_all:
            labels_all = np.concatenate(labels_all, axis=0)
            labels_all[:, 1:] = np.clip(labels_all[:, 1:], 0.0, float(2 * s))
            wh = labels_all[:, 3:5] - labels_all[:, 1:3]
            keep = (wh[:, 0] > 2.0) & (wh[:, 1] > 2.0)
            labels_all = labels_all[keep]
        else:
            labels_all = np.zeros((0, 5), dtype=np.float32)

        out_img = Image.fromarray(mosaic_img).resize((s, s), Image.BILINEAR)
        if labels_all.size > 0:
            labels_all[:, 1:] *= 0.5
            labels_all = _xyxy_abs_to_yolo(labels_all, s)

        return out_img, labels_all

    def __getitem__(self, idx: int):
        p = self.img_paths[idx]

        if self.train and random.random() < self.mosaic_prob and len(self.img_paths) >= 4:
            img, labels = self._load_mosaic(idx)
        else:
            raw_img, raw_labels = self._read_image_and_labels(idx)
            img, labels = self._letterbox_with_labels(raw_img, raw_labels)

        x = torch.from_numpy(np.asarray(img, dtype=np.float32).transpose(2, 0, 1) / 255.0)
        t = torch.tensor(labels, dtype=torch.float32) if labels.size > 0 else torch.zeros((0, 5), dtype=torch.float32)
        return x, t, str(p)


def collate_fn(batch):
    imgs, targets, paths = zip(*batch)
    imgs = torch.stack(imgs, 0)
    return imgs, list(targets), list(paths)

In [ ]:
class ConvBNAct(nn.Module):
    def __init__(self, c1, c2, k=3, s=1, p=None, g=1, act=True):
        super().__init__()
        if p is None:
            p = k // 2
        self.conv = nn.Conv2d(c1, c2, k, s, p, groups=g, bias=False)
        self.bn = nn.BatchNorm2d(c2)
        self.act = nn.SiLU(inplace=True) if act else nn.Identity()

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))


class SIEM(nn.Module):
    """Spatial Information Extraction Module following Eq.(1)(2)(3)."""
    def __init__(self, c_in: int):
        super().__init__()
        c_half = c_in // 2

        # Branch 1 (deeper)
        self.branch1 = nn.Sequential(
            ConvBNAct(c_in, c_half, 3, 1),
            ConvBNAct(c_half, c_half, 3, 1),
            ConvBNAct(c_half, c_half, 3, 1),
        )

        # Branch 2 (shallower)
        self.branch2 = nn.Sequential(
            ConvBNAct(c_in, c_half, 3, 1),
            ConvBNAct(c_half, c_half, 3, 1),
        )

        self.cbs_f1 = ConvBNAct(c_in, c_in, 3, 1)
        self.cbs_attn1 = ConvBNAct(c_in, c_in, 3, 1)
        self.cbs_attn2 = ConvBNAct(c_in, c_in, 3, 1)

        # Eq.(3) uses 5x5 max pooling
        self.pool = nn.MaxPool2d(5, stride=1, padding=2)

        self.conv1x1_1 = nn.Conv2d(c_in, c_in, 1, 1, bias=False)
        self.bn = nn.BatchNorm2d(c_in)
        self.conv1x1_2 = nn.Conv2d(c_in, c_in, 1, 1, bias=False)

    def forward(self, x):
        # Eq.(1): F1 = Cat(branch1, branch2)
        b1 = self.branch1(x)
        b2 = self.branch2(x)
        f1 = torch.cat([b1, b2], dim=1)

        # Eq.(2): F2 = CBS(F1) * Softmax(CBS(CBS(F1)))
        f1_cbs = self.cbs_f1(f1)
        attn_map = F.softmax(self.cbs_attn2(self.cbs_attn1(f1)), dim=1)
        f2 = f1_cbs * attn_map

        # Eq.(3): residual aggregation
        f_out_proj = self.conv1x1_2(self.bn(self.conv1x1_1(f2)))
        out = f_out_proj + f1_cbs + self.pool(x) + x
        return out


class CSPBlock(nn.Module):
    def __init__(self, c1, c2, n=2):
        super().__init__()
        self.cv1 = ConvBNAct(c1, c2, 1, 1)
        self.cv2 = ConvBNAct(c1, c2, 1, 1)
        self.m = nn.Sequential(*[ConvBNAct(c2, c2, 3, 1) for _ in range(n)])
        self.cv3 = ConvBNAct(2 * c2, c2, 1, 1)

    def forward(self, x):
        y1 = self.m(self.cv1(x))
        y2 = self.cv2(x)
        return self.cv3(torch.cat([y1, y2], dim=1))


class MFEFBackbone(nn.Module):
    def __init__(self, base=32):
        super().__init__()
        c1, c2, c3, c4 = base, base * 2, base * 4, base * 8
        self.stem = ConvBNAct(3, c1, 3, 2)            # 320
        self.stage1 = nn.Sequential(
            ConvBNAct(c1, c2, 3, 2),                  # 160
            CSPBlock(c2, c2, n=2),
        )
        self.siem = SIEM(c2)

        self.stage2 = nn.Sequential(
            ConvBNAct(c2, c3, 3, 2),                  # 80
            CSPBlock(c3, c3, n=3),
        )
        self.stage3 = nn.Sequential(
            ConvBNAct(c3, c4, 3, 2),                  # 40
            CSPBlock(c4, c4, n=3),
        )
        self.stage4 = nn.Sequential(
            ConvBNAct(c4, c4 * 2, 3, 2),              # 20
            CSPBlock(c4 * 2, c4 * 2, n=2),
        )

    def forward(self, x):
        x = self.stem(x)
        p2 = self.stage1(x)
        p2 = self.siem(p2)
        p3 = self.stage2(p2)
        p4 = self.stage3(p3)
        p5 = self.stage4(p4)
        return p2, p3, p4, p5

In [ ]:
class AFFM(nn.Module):
    """Adaptive Feature Fusion Module with dynamic per-image weights."""
    def __init__(self, ch_in: List[int], c_out: int):
        super().__init__()
        self.num_inputs = len(ch_in)

        # Step 1: align channels
        self.convs = nn.ModuleList([ConvBNAct(c, c_out, 1, 1) for c in ch_in])

        # Step 2: dynamic fusion weights from concatenated features
        self.weight_conv = nn.Conv2d(c_out * self.num_inputs, self.num_inputs, 1, 1)

        # Final 3x3 convolution after weighted fusion
        self.out_conv = ConvBNAct(c_out, c_out, 3, 1)

    def forward(self, xs: List[torch.Tensor]) -> torch.Tensor:
        ys = [conv(x) for conv, x in zip(self.convs, xs)]

        f_cat = torch.cat(ys, dim=1)
        w = F.softmax(self.weight_conv(f_cat), dim=1)  # [B, num_inputs, H, W]

        out = 0.0
        for i, y in enumerate(ys):
            out = out + y * w[:, i:i + 1, :, :]

        return self.out_conv(out)


class GAFN(nn.Module):
    """Algorithm-1 style progressive fusion from 4 backbone levels."""
    def __init__(self, c2, c3, c4, c5, c=128):
        super().__init__()
        self.fuse_p4 = AFFM([c4, c5], c)
        self.fuse_p3 = AFFM([c3, c], c)
        self.fuse_p2 = AFFM([c2, c], c)

        self.fuse_n3 = AFFM([c, c], c)
        self.fuse_n4 = AFFM([c, c], c)
        self.fuse_n5 = AFFM([c, c], c)

        self.p5_lat = ConvBNAct(c5, c, 1, 1)
        self.ds = nn.AvgPool2d(2, 2)

    def forward(self, p2, p3, p4, p5):
        p5u = F.interpolate(p5, size=p4.shape[-2:], mode='nearest')
        m4 = self.fuse_p4([p4, p5u])

        m4u = F.interpolate(m4, size=p3.shape[-2:], mode='nearest')
        m3 = self.fuse_p3([p3, m4u])

        m3u = F.interpolate(m3, size=p2.shape[-2:], mode='nearest')
        m2 = self.fuse_p2([p2, m3u])

        # Progressive down path
        n3 = self.fuse_n3([m3, self.ds(m2)])
        n4 = self.fuse_n4([m4, self.ds(n3)])
        p5r = self.p5_lat(p5)
        n5 = self.fuse_n5([self.ds(n4), F.interpolate(p5r, size=self.ds(n4).shape[-2:], mode='nearest')])

        return m2, n3, n4, n5


class IDetectLikeHead(nn.Module):
    def __init__(self, c=128, num_classes=10, num_anchors=3):
        super().__init__()
        self.num_classes = num_classes
        self.num_outputs = num_classes + 5
        self.num_anchors = num_anchors
        self.pred2 = nn.Conv2d(c, num_anchors * self.num_outputs, 1)
        self.pred3 = nn.Conv2d(c, num_anchors * self.num_outputs, 1)
        self.pred4 = nn.Conv2d(c, num_anchors * self.num_outputs, 1)
        self.pred5 = nn.Conv2d(c, num_anchors * self.num_outputs, 1)

    def _reshape(self, x):
        b, _, h, w = x.shape
        x = x.view(b, self.num_anchors, self.num_outputs, h, w)
        return x.permute(0, 1, 3, 4, 2).contiguous()  # [B, A, H, W, 5+nc]

    def forward(self, feats):
        m2, n3, n4, n5 = feats
        return [
            self._reshape(self.pred2(m2)),
            self._reshape(self.pred3(n3)),
            self._reshape(self.pred4(n4)),
            self._reshape(self.pred5(n5)),
        ]


class MFEFNet(nn.Module):
    def __init__(self, num_classes=10, base=32, neck_c=128):
        super().__init__()
        self.backbone = MFEFBackbone(base=base)
        c2, c3, c4, c5 = base * 2, base * 4, base * 8, base * 16
        self.neck = GAFN(c2, c3, c4, c5, c=neck_c)
        self.head = IDetectLikeHead(c=neck_c, num_classes=num_classes)

    def forward(self, x):
        feats = self.backbone(x)
        fused = self.neck(*feats)
        return self.head(fused)

In [ ]:
def mpdiou_loss(pred_xyxy: torch.Tensor, tgt_xyxy: torch.Tensor, eps: float = 1e-7) -> torch.Tensor:
    """MPDIoU: IoU - normalized distance(top-left, bottom-right)."""
    px1, py1, px2, py2 = pred_xyxy.unbind(-1)
    tx1, ty1, tx2, ty2 = tgt_xyxy.unbind(-1)

    inter_x1 = torch.maximum(px1, tx1)
    inter_y1 = torch.maximum(py1, ty1)
    inter_x2 = torch.minimum(px2, tx2)
    inter_y2 = torch.minimum(py2, ty2)

    inter_w = (inter_x2 - inter_x1).clamp(min=0)
    inter_h = (inter_y2 - inter_y1).clamp(min=0)
    inter = inter_w * inter_h

    p_area = (px2 - px1).clamp(min=0) * (py2 - py1).clamp(min=0)
    t_area = (tx2 - tx1).clamp(min=0) * (ty2 - ty1).clamp(min=0)
    union = p_area + t_area - inter + eps
    iou = inter / union

    ex1 = torch.minimum(px1, tx1)
    ey1 = torch.minimum(py1, ty1)
    ex2 = torch.maximum(px2, tx2)
    ey2 = torch.maximum(py2, ty2)
    diag2 = (ex2 - ex1).pow(2) + (ey2 - ey1).pow(2) + eps

    d_tl = (px1 - tx1).pow(2) + (py1 - ty1).pow(2)
    d_br = (px2 - tx2).pow(2) + (py2 - ty2).pow(2)

    mpdiou = iou - (d_tl + d_br) / diag2
    return 1.0 - mpdiou.clamp(min=-1.0, max=1.0)


def decode_boxes_from_logits(pred: torch.Tensor, anchors_wh: Optional[torch.Tensor] = None) -> torch.Tensor:
    """Decode [B, A, H, W, 5+nc] -> normalized xyxy in [0,1]."""
    _, a, h, w, _ = pred.shape
    device = pred.device
    dtype = pred.dtype

    gy, gx = torch.meshgrid(
        torch.arange(h, device=device, dtype=dtype),
        torch.arange(w, device=device, dtype=dtype),
        indexing='ij'
    )
    gx = gx.view(1, 1, h, w)
    gy = gy.view(1, 1, h, w)

    tx = pred[..., 0]
    ty = pred[..., 1]
    tw = pred[..., 2]
    th = pred[..., 3]

    cx = (torch.sigmoid(tx) + gx) / max(1.0, float(w))
    cy = (torch.sigmoid(ty) + gy) / max(1.0, float(h))

    if anchors_wh is not None:
        anc = anchors_wh.to(device=device, dtype=dtype)
        bw = torch.sigmoid(tw) * anc[:, 0].view(1, a, 1, 1)
        bh = torch.sigmoid(th) * anc[:, 1].view(1, a, 1, 1)
    else:
        bw = torch.sigmoid(tw)
        bh = torch.sigmoid(th)

    x1 = (cx - bw / 2.0).clamp(0.0, 1.0)
    y1 = (cy - bh / 2.0).clamp(0.0, 1.0)
    x2 = (cx + bw / 2.0).clamp(0.0, 1.0)
    y2 = (cy + bh / 2.0).clamp(0.0, 1.0)

    return torch.stack([x1, y1, x2, y2], dim=-1)


def _pick_level_by_area(area: float, n_levels: int) -> int:
    """Simple scale assignment from small to large boxes."""
    bins = [0.02, 0.08, 0.20]
    if n_levels <= 1:
        return 0
    if n_levels == 2:
        return 0 if area < bins[1] else 1
    if n_levels == 3:
        return 0 if area < bins[0] else (1 if area < bins[2] else 2)
    if area < bins[0]:
        return 0
    if area < bins[1]:
        return 1
    if area < bins[2]:
        return 2
    return 3


def _best_anchor_index(gt_w: float, gt_h: float, anchors_wh: torch.Tensor) -> int:
    """Pick anchor by minimizing max(width_ratio, height_ratio)."""
    gt = torch.tensor([gt_w, gt_h], dtype=anchors_wh.dtype, device=anchors_wh.device).clamp(min=1e-6)
    anc = anchors_wh.clamp(min=1e-6)
    ratio = torch.maximum(gt[None, :] / anc, anc / gt[None, :])
    score = ratio.max(dim=1).values
    return int(torch.argmin(score).item())


DEFAULT_ANCHORS_PX = [
    [(10, 13), (16, 30), (33, 23)],
    [(30, 61), (62, 45), (59, 119)],
    [(116, 90), (156, 198), (373, 326)],
    [(200, 280), (300, 380), (420, 520)],
]


@dataclass
class TrainConfig:
    epochs: int = 300
    batch_size: int = 64
    lr0: float = 0.020
    momentum: float = 0.937
    weight_decay: float = 5e-4
    warmup_epochs: int = 3
    min_lr_ratio: float = 0.01
    mosaic_prob: float = 0.8


class MFEFNetLoss(nn.Module):
    """Total loss = 0.3*MPDIoU + 0.05*BCE_reg + 0.7*BCE_obj."""
    def __init__(self, num_classes: int = 10, img_size: int = 640, anchors_px: Optional[List[List[Tuple[int, int]]]] = None):
        super().__init__()
        self.num_classes = num_classes
        self.bce = nn.BCEWithLogitsLoss(reduction='mean')

        anchors_px = anchors_px if anchors_px is not None else DEFAULT_ANCHORS_PX
        self.anchors_wh = [torch.tensor(a, dtype=torch.float32) / float(img_size) for a in anchors_px]

    def _allocate_targets(self, preds: List[torch.Tensor]):
        obj_t, cls_t, box_t, pos_mask = [], [], [], []
        for p in preds:
            b, a, h, w, _ = p.shape
            device = p.device
            obj_t.append(torch.zeros((b, a, h, w), device=device, dtype=p.dtype))
            cls_t.append(torch.zeros((b, a, h, w, self.num_classes), device=device, dtype=p.dtype))
            box_t.append(torch.zeros((b, a, h, w, 4), device=device, dtype=p.dtype))
            pos_mask.append(torch.zeros((b, a, h, w), device=device, dtype=torch.bool))
        return obj_t, cls_t, box_t, pos_mask

    def _build_dense_targets(self, preds: List[torch.Tensor], targets: List[torch.Tensor]):
        obj_t, cls_t, box_t, pos_mask = self._allocate_targets(preds)
        n_levels = len(preds)

        for b_ix, t in enumerate(targets):
            if t.numel() == 0:
                continue
            for row in t:
                cls_id = int(row[0].item())
                cx = float(row[1].item())
                cy = float(row[2].item())
                bw = float(row[3].item())
                bh = float(row[4].item())

                if cls_id < 0 or cls_id >= self.num_classes:
                    continue
                if bw <= 0.0 or bh <= 0.0:
                    continue

                level = min(_pick_level_by_area(bw * bh, n_levels), n_levels - 1)
                _, a, h, w, _ = preds[level].shape

                anchors_l = self.anchors_wh[min(level, len(self.anchors_wh) - 1)].to(preds[level].device)
                if anchors_l.shape[0] != a:
                    anchor_ix = 0
                else:
                    anchor_ix = _best_anchor_index(bw, bh, anchors_l)

                gx = int(max(0, min(w - 1, math.floor(cx * w))))
                gy = int(max(0, min(h - 1, math.floor(cy * h))))

                x1 = max(0.0, cx - bw / 2.0)
                y1 = max(0.0, cy - bh / 2.0)
                x2 = min(1.0, cx + bw / 2.0)
                y2 = min(1.0, cy + bh / 2.0)

                obj_t[level][b_ix, anchor_ix, gy, gx] = 1.0
                cls_t[level][b_ix, anchor_ix, gy, gx, cls_id] = 1.0
                box_t[level][b_ix, anchor_ix, gy, gx] = torch.tensor(
                    [x1, y1, x2, y2],
                    device=preds[level].device,
                    dtype=preds[level].dtype,
                )
                pos_mask[level][b_ix, anchor_ix, gy, gx] = True

        return obj_t, cls_t, box_t, pos_mask

    def forward(self, preds: List[torch.Tensor], targets: List[torch.Tensor]) -> Dict[str, torch.Tensor]:
        obj_t, cls_t, box_t, pos_mask = self._build_dense_targets(preds, targets)

        l_box = torch.zeros((), device=preds[0].device)
        l_cls = torch.zeros((), device=preds[0].device)
        l_obj = torch.zeros((), device=preds[0].device)

        for s, p in enumerate(preds):
            obj_logits = p[..., 4]
            cls_logits = p[..., 5:5 + self.num_classes]
            anchors_l = self.anchors_wh[min(s, len(self.anchors_wh) - 1)]
            pred_boxes = decode_boxes_from_logits(p, anchors_wh=anchors_l)

            l_obj = l_obj + self.bce(obj_logits, obj_t[s])

            pm = pos_mask[s]
            if pm.any():
                l_box = l_box + mpdiou_loss(pred_boxes[pm], box_t[s][pm]).mean()
                l_cls = l_cls + self.bce(cls_logits[pm], cls_t[s][pm])

        num_scales = max(1, len(preds))
        l_box = l_box / num_scales
        l_cls = l_cls / num_scales
        l_obj = l_obj / num_scales

        total = 0.3 * l_box + 0.05 * l_cls + 0.7 * l_obj
        return {'loss': total, 'l_box': l_box, 'l_cls': l_cls, 'l_obj': l_obj}


CFG = TrainConfig()
print(CFG)


def build_optimizer(model: nn.Module, cfg: TrainConfig) -> torch.optim.Optimizer:
    return torch.optim.SGD(
        model.parameters(),
        lr=cfg.lr0,
        momentum=cfg.momentum,
        weight_decay=cfg.weight_decay,
        nesterov=True,
    )


def build_scheduler(optimizer: torch.optim.Optimizer, cfg: TrainConfig, steps_per_epoch: int):
    """Warmup (first cfg.warmup_epochs) + cosine decay to cfg.min_lr_ratio."""
    steps_per_epoch = max(1, int(steps_per_epoch))
    total_steps = max(1, cfg.epochs * steps_per_epoch)
    warmup_steps = max(1, cfg.warmup_epochs * steps_per_epoch)

    def lr_lambda(step: int):
        if step < warmup_steps:
            return float(step + 1) / float(warmup_steps)
        progress = (step - warmup_steps) / float(max(1, total_steps - warmup_steps))
        cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
        return cfg.min_lr_ratio + (1.0 - cfg.min_lr_ratio) * cosine

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)

TrainConfig(epochs=300, batch_size=64, lr0=0.02, momentum=0.937, weight_decay=0.0005, warmup_epochs=3, min_lr_ratio=0.01, mosaic_prob=0.8)


In [ ]:
def evaluate_coco_stub(coco_gt_json: Path, coco_pred_json: Path):
    if not HAS_COCO:
        raise RuntimeError('pycocotools is not installed. Run: %pip install pycocotools')

    # Dynamic import to avoid static unresolved-import warnings in the editor.
    coco_mod = importlib.import_module('pycocotools.coco')
    cocoeval_mod = importlib.import_module('pycocotools.cocoeval')
    COCO = coco_mod.COCO
    COCOeval = cocoeval_mod.COCOeval

    coco_gt = COCO(str(coco_gt_json))
    coco_dt = coco_gt.loadRes(str(coco_pred_json))

    evaluator = COCOeval(coco_gt, coco_dt, iouType='bbox')
    evaluator.evaluate()
    evaluator.accumulate()
    evaluator.summarize()

    out = {
        'mAP50_95': float(evaluator.stats[0]),
        'mAP50': float(evaluator.stats[1]),
        'APS': float(evaluator.stats[3]),
        'APM': float(evaluator.stats[4]),
        'APL': float(evaluator.stats[5]),
    }
    return out


def evaluate_coco_metrics(coco_gt_json: Path, coco_pred_json: Path) -> Dict[str, float]:
    """Return standard COCO detection metrics."""
    return evaluate_coco_stub(coco_gt_json, coco_pred_json)


def sanity_model_forward(num_classes: int = 10):
    model = MFEFNet(num_classes=num_classes).to(DEVICE)
    x = torch.randn(2, 3, 640, 640, device=DEVICE)
    with torch.no_grad():
        y = model(x)
    print('Num output scales:', len(y))
    for i, p in enumerate(y):
        print(f'scale-{i}:', tuple(p.shape))


def _xywhn_to_xyxy_abs(cx: float, cy: float, bw: float, bh: float, w: int, h: int) -> Tuple[float, float, float, float]:
    x1 = (cx - bw / 2.0) * w
    y1 = (cy - bh / 2.0) * h
    x2 = (cx + bw / 2.0) * w
    y2 = (cy + bh / 2.0) * h
    return x1, y1, x2, y2


def build_image_id_map(image_list_file: Path) -> Dict[str, int]:
    img_paths = [Path(x.strip()) for x in image_list_file.read_text(encoding='utf-8').splitlines() if x.strip()]
    return {str(p.resolve()): i + 1 for i, p in enumerate(img_paths)}


def build_coco_gt_from_yolo(
    image_list_file: Path,
    labels_dir: Path,
    class_names: List[str],
    output_json: Path,
) -> Path:
    """Build COCO GT json from YOLO txt labels."""
    img_paths = [Path(x.strip()) for x in image_list_file.read_text(encoding='utf-8').splitlines() if x.strip()]

    images = []
    annotations = []
    categories = [{'id': i + 1, 'name': n, 'supercategory': 'object'} for i, n in enumerate(class_names)]

    ann_id = 1
    for img_id, p in enumerate(img_paths, start=1):
        with Image.open(p) as im:
            w, h = im.size

        images.append({'id': img_id, 'file_name': p.name, 'width': w, 'height': h})

        label_path = labels_dir / f'{p.stem}.txt'
        if not label_path.exists():
            continue

        for ln in label_path.read_text(encoding='utf-8').splitlines():
            if not ln.strip():
                continue
            cls_s, cx_s, cy_s, bw_s, bh_s = ln.split()
            cls_id = int(cls_s)
            cx, cy, bw, bh = float(cx_s), float(cy_s), float(bw_s), float(bh_s)

            x1, y1, x2, y2 = _xywhn_to_xyxy_abs(cx, cy, bw, bh, w, h)
            x1 = max(0.0, min(float(w), x1))
            y1 = max(0.0, min(float(h), y1))
            x2 = max(0.0, min(float(w), x2))
            y2 = max(0.0, min(float(h), y2))

            bw_abs = max(0.0, x2 - x1)
            bh_abs = max(0.0, y2 - y1)
            if bw_abs <= 0.0 or bh_abs <= 0.0:
                continue

            annotations.append(
                {
                    'id': ann_id,
                    'image_id': img_id,
                    'category_id': cls_id + 1,
                    'bbox': [x1, y1, bw_abs, bh_abs],
                    'area': bw_abs * bh_abs,
                    'iscrowd': 0,
                }
            )
            ann_id += 1

    coco = {'images': images, 'annotations': annotations, 'categories': categories}
    output_json.parent.mkdir(parents=True, exist_ok=True)
    output_json.write_text(json.dumps(coco, ensure_ascii=False), encoding='utf-8')
    return output_json


def _box_iou_xyxy(box: torch.Tensor, boxes: torch.Tensor, eps: float = 1e-7) -> torch.Tensor:
    ix1 = torch.maximum(box[0], boxes[:, 0])
    iy1 = torch.maximum(box[1], boxes[:, 1])
    ix2 = torch.minimum(box[2], boxes[:, 2])
    iy2 = torch.minimum(box[3], boxes[:, 3])

    iw = (ix2 - ix1).clamp(min=0)
    ih = (iy2 - iy1).clamp(min=0)
    inter = iw * ih

    a1 = (box[2] - box[0]).clamp(min=0) * (box[3] - box[1]).clamp(min=0)
    a2 = (boxes[:, 2] - boxes[:, 0]).clamp(min=0) * (boxes[:, 3] - boxes[:, 1]).clamp(min=0)
    return inter / (a1 + a2 - inter + eps)


def _nms_xyxy(boxes: torch.Tensor, scores: torch.Tensor, iou_thres: float = 0.65) -> torch.Tensor:
    if boxes.numel() == 0:
        return torch.zeros((0,), dtype=torch.long, device=boxes.device)

    order = torch.argsort(scores, descending=True)
    keep = []
    while order.numel() > 0:
        i = int(order[0].item())
        keep.append(i)
        if order.numel() == 1:
            break

        rest = order[1:]
        ious = _box_iou_xyxy(boxes[i], boxes[rest])
        order = rest[ious <= iou_thres]

    return torch.tensor(keep, dtype=torch.long, device=boxes.device)


def _letterbox_to_original_xyxy(
    box_xyxy_norm: torch.Tensor,
    orig_w: int,
    orig_h: int,
    input_size: int,
) -> Tuple[float, float, float, float]:
    x1, y1, x2, y2 = (box_xyxy_norm * float(input_size)).tolist()

    scale = min(float(input_size) / float(orig_w), float(input_size) / float(orig_h))
    nw, nh = int(round(orig_w * scale)), int(round(orig_h * scale))
    pad_w = (input_size - nw) / 2.0
    pad_h = (input_size - nh) / 2.0

    x1 = (x1 - pad_w) / max(scale, 1e-7)
    y1 = (y1 - pad_h) / max(scale, 1e-7)
    x2 = (x2 - pad_w) / max(scale, 1e-7)
    y2 = (y2 - pad_h) / max(scale, 1e-7)

    x1 = max(0.0, min(float(orig_w), x1))
    y1 = max(0.0, min(float(orig_h), y1))
    x2 = max(0.0, min(float(orig_w), x2))
    y2 = max(0.0, min(float(orig_h), y2))
    return x1, y1, x2, y2


def _collect_batch_dets(
    preds: List[torch.Tensor],
    conf_thres: float,
    iou_thres: float,
    max_det: int,
    num_classes: int,
) -> List[torch.Tensor]:
    """Return list of detections per image: [N, 6] => x1,y1,x2,y2,score,cls."""
    batch_size = preds[0].shape[0]
    dets_per_im = []

    for b in range(batch_size):
        boxes_all = []
        scores_all = []
        cls_all = []

        for p in preds:
            p_b = p[b:b + 1]
            boxes = decode_boxes_from_logits(p_b)[0].reshape(-1, 4)
            obj = torch.sigmoid(p_b[0, ..., 4]).reshape(-1)
            cls_prob = torch.sigmoid(p_b[0, ..., 5:5 + num_classes]).reshape(-1, num_classes)
            cls_score, cls_idx = torch.max(cls_prob, dim=1)
            score = obj * cls_score

            m = score >= conf_thres
            if m.any():
                boxes_all.append(boxes[m])
                scores_all.append(score[m])
                cls_all.append(cls_idx[m].float())

        if not boxes_all:
            dets_per_im.append(torch.zeros((0, 6), device=preds[0].device, dtype=preds[0].dtype))
            continue

        boxes = torch.cat(boxes_all, dim=0)
        scores = torch.cat(scores_all, dim=0)
        clses = torch.cat(cls_all, dim=0)

        keep_all = []
        for c in clses.unique():
            cm = clses == c
            keep_c = _nms_xyxy(boxes[cm], scores[cm], iou_thres=iou_thres)
            orig_idx = torch.where(cm)[0][keep_c]
            keep_all.append(orig_idx)

        keep = torch.cat(keep_all) if keep_all else torch.zeros((0,), dtype=torch.long, device=boxes.device)
        if keep.numel() > 0:
            keep = keep[torch.argsort(scores[keep], descending=True)]
            keep = keep[:max_det]

        det = torch.cat([boxes[keep], scores[keep, None], clses[keep, None]], dim=1) if keep.numel() > 0 else torch.zeros((0, 6), device=boxes.device)
        dets_per_im.append(det)

    return dets_per_im


def export_predictions_to_coco_json(
    model: nn.Module,
    image_list_file: Path,
    labels_dir: Path,
    output_json: Path,
    img_size: int = 640,
    batch_size: int = 8,
    conf_thres: float = 0.001,
    iou_thres: float = 0.65,
    max_det: int = 300,
    num_classes: int = 10,
) -> Path:
    """Run model on image list and export COCO detection json."""
    ds = YoloTxtDataset(image_list_file=image_list_file, labels_dir=labels_dir, img_size=img_size)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=0, collate_fn=collate_fn)

    image_id_map = build_image_id_map(image_list_file)
    records = []

    model.eval()
    with torch.no_grad():
        for imgs, _, paths in dl:
            imgs = imgs.to(DEVICE)
            preds = model(imgs)
            dets_batch = _collect_batch_dets(
                preds=preds,
                conf_thres=conf_thres,
                iou_thres=iou_thres,
                max_det=max_det,
                num_classes=num_classes,
            )

            for dets, p_str in zip(dets_batch, paths):
                p = Path(p_str)
                img_id = image_id_map.get(str(p.resolve()))
                if img_id is None:
                    continue

                with Image.open(p) as im:
                    ow, oh = im.size

                for d in dets:
                    x1, y1, x2, y2 = _letterbox_to_original_xyxy(d[:4], ow, oh, img_size)
                    bw = max(0.0, x2 - x1)
                    bh = max(0.0, y2 - y1)
                    if bw <= 0.0 or bh <= 0.0:
                        continue

                    records.append(
                        {
                            'image_id': int(img_id),
                            'category_id': int(d[5].item()) + 1,
                            'bbox': [float(x1), float(y1), float(bw), float(bh)],
                            'score': float(d[4].item()),
                        }
                    )

    output_json.parent.mkdir(parents=True, exist_ok=True)
    output_json.write_text(json.dumps(records, ensure_ascii=False), encoding='utf-8')
    return output_json


def evaluate_visdrone_val_from_model(
    model: nn.Module,
    output_dir: Path,
    img_size: int = 640,
    batch_size: int = 8,
    conf_thres: float = 0.001,
    iou_thres: float = 0.65,
    max_det: int = 300,
) -> Dict[str, float]:
    """End-to-end evaluation for VisDrone val with standard COCO metrics."""
    val_img_list = P.workdir / 'visdrone' / 'val_images.txt'
    val_lbl_dir = P.workdir / 'visdrone' / 'val' / 'labels'

    gt_json = output_dir / 'visdrone_val_gt_coco.json'
    pred_json = output_dir / 'visdrone_val_pred_coco.json'

    build_coco_gt_from_yolo(
        image_list_file=val_img_list,
        labels_dir=val_lbl_dir,
        class_names=VISDRONE_NAMES,
        output_json=gt_json,
    )

    export_predictions_to_coco_json(
        model=model,
        image_list_file=val_img_list,
        labels_dir=val_lbl_dir,
        output_json=pred_json,
        img_size=img_size,
        batch_size=batch_size,
        conf_thres=conf_thres,
        iou_thres=iou_thres,
        max_det=max_det,
        num_classes=len(VISDRONE_NAMES),
    )

    return evaluate_coco_metrics(gt_json, pred_json)

## 6) Pipeline Entrypoints (Write-Only)

Cell tiếp theo định nghĩa toàn bộ entrypoints theo thứ tự thực thi chuẩn:
1. Data inspection
2. Data conversion
3. DataLoader build
4. Model/Loss/Optimizer setup
5. Full training loop (theo cấu hình 300 epochs mặc định)
6. Evaluation COCO (từ json hoặc trực tiếp từ model)

Mặc định không tự chạy để tránh tốn tài nguyên và giữ notebook ở trạng thái write-only.

In [ ]:
def build_visdrone_train_loader(
    batch_size: int = 64,
    img_size: int = 640,
    shuffle: bool = True,
    train: bool = True,
    mosaic_prob: float = 0.8,
    num_workers: int = 8,
):
    ds = YoloTxtDataset(
        image_list_file=P.workdir / 'visdrone' / 'train_images.txt',
        labels_dir=P.workdir / 'visdrone' / 'train' / 'labels',
        img_size=img_size,
        train=train,
        mosaic_prob=mosaic_prob if train else 0.0,
    )
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=True,
        persistent_workers=True if train else False,
        collate_fn=collate_fn,
    )


def setup_training(num_classes: int = 10, cfg: Optional[TrainConfig] = None):
    cfg = cfg or CFG
    model = MFEFNet(num_classes=num_classes).to(DEVICE)
    optimizer = build_optimizer(model, cfg)
    loss_fn = MFEFNetLoss(num_classes=num_classes)
    scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE == 'cuda'))
    return model, optimizer, loss_fn, scaler


def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    loss_fn: nn.Module,
    scaler: torch.amp.GradScaler,
    scheduler: Optional[torch.optim.lr_scheduler.LambdaLR] = None,
    max_steps: Optional[int] = None,
) -> Dict[str, float]:
    model.train()
    sum_loss, sum_box, sum_cls, sum_obj = 0.0, 0.0, 0.0, 0.0
    steps = 0

    use_amp = DEVICE == 'cuda'
    autocast_device = 'cuda' if use_amp else 'cpu'

    for imgs, targets, _ in loader:
        imgs = imgs.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(device_type=autocast_device, enabled=use_amp):
            preds = model(imgs)
            loss_dict = loss_fn(preds, targets)

        scaler.scale(loss_dict['loss']).backward()

        prev_scale = scaler.get_scale()
        scaler.step(optimizer)
        scaler.update()

        if scheduler is not None and scaler.get_scale() >= prev_scale:
            scheduler.step()

        sum_loss += float(loss_dict['loss'].detach().cpu())
        sum_box += float(loss_dict['l_box'].detach().cpu())
        sum_cls += float(loss_dict['l_cls'].detach().cpu())
        sum_obj += float(loss_dict['l_obj'].detach().cpu())
        steps += 1

        if max_steps is not None and steps >= max_steps:
            break

    den = max(1, steps)
    return {
        'loss': sum_loss / den,
        'l_box': sum_box / den,
        'l_cls': sum_cls / den,
        'l_obj': sum_obj / den,
        'steps': float(steps),
        'lr': float(optimizer.param_groups[0]['lr']),
    }


def train_model(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    loss_fn: nn.Module,
    scaler: torch.amp.GradScaler,
    cfg: TrainConfig,
    max_steps_per_epoch: Optional[int] = None,
    checkpoint_dir: Optional[Path] = None,
    checkpoint_name: str = 'best_model.pt',
) -> List[Dict[str, float]]:
    """Main training loop with warmup + cosine LR schedule and best-checkpoint saving."""
    history: List[Dict[str, float]] = []

    if max_steps_per_epoch is not None:
        steps_per_epoch = max(1, int(max_steps_per_epoch))
    else:
        steps_per_epoch = max(1, len(loader))
    print(1)
    scheduler = build_scheduler(optimizer, cfg, steps_per_epoch=steps_per_epoch)
    print(2)
    if checkpoint_dir is None:
        checkpoint_dir = P.workdir / 'checkpoints'
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    ckpt_path = checkpoint_dir / checkpoint_name

    best_loss = float('inf')
    best_epoch = 0
    print(3)
    for epoch in range(cfg.epochs):
        log = train_one_epoch(
            model=model,
            loader=loader,
            optimizer=optimizer,
            loss_fn=loss_fn,
            scaler=scaler,
            scheduler=scheduler,
            max_steps=max_steps_per_epoch,
        )
        log['epoch'] = float(epoch + 1)
        history.append(log)

        current_loss = float(log['loss'])
        if current_loss < best_loss:
            best_loss = current_loss
            best_epoch = epoch + 1
            torch.save(
                {
                    'epoch': best_epoch,
                    'best_loss': best_loss,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'cfg': cfg.__dict__,
                },
                ckpt_path,
            )

        if (epoch + 1) % 10 == 0 or epoch == 0 or (epoch + 1) == cfg.epochs:
            print(
                f"[Epoch {epoch + 1}/{cfg.epochs}] "
                f"lr={log['lr']:.6f} "
                f"loss={log['loss']:.4f} box={log['l_box']:.4f} "
                f"cls={log['l_cls']:.4f} obj={log['l_obj']:.4f} "
                f"best={best_loss:.4f}@{best_epoch}"
            )

    print(f'Best checkpoint saved: {ckpt_path} (epoch={best_epoch}, loss={best_loss:.6f})')
    return history


def train_pipeline(
    run_data_prep: bool = False,
    num_classes: int = 10,
    max_steps_per_epoch: Optional[int] = None,
    cfg: Optional[TrainConfig] = None,
    checkpoint_dir: Optional[Path] = None,
):
    """Pipeline:
    1) inspect data structure
    2) optional conversion to YOLO labels
    3) build train loader (+ Mosaic)
    4) setup model/loss/optimizer (SGD + momentum)
    5) train for cfg.epochs with warmup + cosine schedule
    6) save best checkpoint by training loss
    """

    cfg = cfg or CFG
    inspect_raw_structure()

    prep_stats = None
    if run_data_prep:
        prep_stats = prepare_data()
    print(1)
    loader = build_visdrone_train_loader(
        batch_size=cfg.batch_size,
        img_size=640,
        shuffle=True,
        train=True,
        mosaic_prob=cfg.mosaic_prob,
        num_workers=8,
    )
    print(2)
    model, optimizer, loss_fn, scaler = setup_training(num_classes=num_classes, cfg=cfg)

    checkpoint_dir = checkpoint_dir or (P.workdir / 'checkpoints')
    print(3)
    history = train_model(
        model=model,
        loader=loader,
        optimizer=optimizer,
        loss_fn=loss_fn,
        scaler=scaler,
        cfg=cfg,
        max_steps_per_epoch=max_steps_per_epoch,
        checkpoint_dir=checkpoint_dir,
    )
    print(4)
    return {
        'cfg': cfg,
        'prep_stats': prep_stats,
        'history': history,
        'model': model,
        'best_checkpoint': checkpoint_dir / 'best_model.pt',
    }


def eval_pipeline(coco_gt_json: Path, coco_pred_json: Path) -> Dict[str, float]:
    """COCO evaluation from prepared json files.

    Metrics: mAP50, mAP50_95, APS, APM, APL.
    """
    return evaluate_coco_metrics(coco_gt_json, coco_pred_json)


def eval_pipeline_from_model(
    model: nn.Module,
    output_dir: Path,
    img_size: int = 640,
    batch_size: int = 8,
    conf_thres: float = 0.001,
    iou_thres: float = 0.65,
    max_det: int = 300,
) -> Dict[str, float]:
    """End-to-end COCO evaluation directly from model predictions on VisDrone val."""
    return evaluate_visdrone_val_from_model(
        model=model,
        output_dir=output_dir,
        img_size=img_size,
        batch_size=batch_size,
        conf_thres=conf_thres,
        iou_thres=iou_thres,
        max_det=max_det,
    )


# Manual usage (intentionally not executed):
# inspect_raw_structure()
# stats = prepare_data()
# print(stats)
# cfg = colab_train_config()
# out = train_pipeline(run_data_prep=False, num_classes=10, cfg=cfg)
# print(out['history'][-1])
# print('best checkpoint:', out['best_checkpoint'])
# metrics = eval_pipeline(Path('instances_val.json'), Path('predictions.json'))
# print(metrics)
# metrics2 = eval_pipeline_from_model(out['model'], P.workdir / 'eval')
# print(metrics2)

In [ ]:
# Colab-friendly training preset (optional)
def colab_train_config() -> TrainConfig:
    """A safer high-throughput preset for Colab A100."""
    return TrainConfig(
        epochs=100,
        batch_size=64,
        lr0=0.02,
        momentum=0.937,
        weight_decay=5e-4,
        warmup_epochs=3,
        min_lr_ratio=0.01,
        mosaic_prob=0.8,
    )

# Example manual usage for Colab:
cfg = colab_train_config()


In [ ]:
# stats = prepare_data()  # run once

In [ ]:
out = train_pipeline(run_data_prep=False, num_classes=10, cfg=cfg)
metrics = eval_pipeline_from_model(out['model'], P.workdir / 'eval')
print(metrics)

VisDrone annotation sample: 9999998_00171_d_0000140.txt
651,758,132,55,1,4,0,0
839,756,115,58,1,5,0,0
1255,773,126,58,1,4,0,0
1589,779,114,55,1,4,0,0
1664,577,169,61,1,4,0,0
1255,621,166,53,1,4,0,0
1316,735,57,31,0,0,0,0
1394,829,41,28,1,3,0,0

UAVDT annotation sample: M1302_img000054.jpg.json
Keys: ['description', 'tags', 'size', 'objects']
Image size: {'height': 540, 'width': 1024}
First object keys: ['id', 'classId', 'objectId', 'description', 'geometryType', 'labelerLogin', 'createdAt', 'updatedAt', 'tags', 'classTitle', 'points']
First object class: car
First object points: {'exterior': [[157, 433], [229, 485]], 'interior': []}
1
2
3
1
2
3


/tmp/ipykernel_10728/3563990864.py:66: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


[Epoch 1/100] lr=0.006732 loss=0.6320 box=0.5601 cls=0.4250 obj=0.6324 best=0.6320@1


KeyboardInterrupt: 